<a href="https://colab.research.google.com/github/taijarals/trade_bot/blob/main/trade_bit_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install openai python-dotenv

## Modularização do Código

Para tornar o código mais organizado e reutilizável, vamos dividi-lo em módulos (`.py` files):

1.  **`foxbit_api.py`**: Conterá as funções para interagir com a API da Foxbit, incluindo a geração de assinatura e a função genérica de chamada à API.
2.  **`market_data_fetcher.py`**: Terá as funções específicas para buscar dados de mercado, como `ticker`, `candlesticks` e `orderbook`.
3.  **`market_analysis.py`**: Incluirá as funções para resumir e analisar os dados de mercado, construindo o contexto para a IA.
4.  **`ai_integration.py`**: Cuidará da montagem do prompt e da interação com a API do modelo de linguagem (OpenRouter/OpenAI).

Vamos começar criando esses arquivos e depois atualizaremos o notebook para usá-los.

In [ ]:
%%writefile foxbit_api.py

import time
import hmac
import hashlib
import json
import requests
from urllib.parse import urlencode

# ==============================================================
# 🧩 CONEXÃO PRIVADA COM A API FOXBIT (EXCHANGE)
# ==============================================================

# ALERTA DE SEGURANÇA: Para produção, considere carregar estas chaves de variáveis de ambiente
# ou do gerenciador de segredos do Colab (userdata) e nunca hardcode-as.
from google.colab import userdata
FOXBIT_KEY = userdata.get('FOXBIT_KEY')
FOXBIT_SECRET = userdata.get('FOXBIT_SECRET')

API_BASE = "https://api.foxbit.com.br"



def gerar_assinatura(api_secret, method, path, params=None, body=""):
    """Gera a assinatura HMAC e cabeçalhos exigidos pela Foxbit."""

    timestamp = str(int(time.time() * 1000))

    queryString = urlencode(params) if params else ''

    rawBody = json.dumps(body) if body else ''

    preHash = f"{timestamp}{method.upper()}{path}{queryString}{rawBody}"

    assinatura = hmac.new(
        api_secret.encode(), preHash.encode(), hashlib.sha256
    ).hexdigest()

    headers = {
        "X-FB-ACCESS-KEY": FOXBIT_KEY,
        "X-FB-ACCESS-TIMESTAMP": timestamp,
        "X-FB-ACCESS-SIGNATURE": assinatura,
        "Content-Type": "application/json",
    }
    return headers


def chamada_api_privada(method, endpoint, payload=None, params=None):
    url = f"{API_BASE}{endpoint}"

    headers = gerar_assinatura(API_SECRET, method, endpoint, params, payload)

    try:
        response = requests.request(method, url, headers=headers, json=payload, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Erro ao acessar API: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print("📩 Resposta da API:", e.response.text)
        return None


In [ ]:
%%writefile market_data_fetcher.py

import requests
import pandas as pd
from foxbit_api import API_BASE # Importa a base da API


def obter_ticker(symbol):
    """Obtém o preço atual de mercado (último preço) para o par informado."""
    endpoint = f"/rest/v3/markets/{symbol}/ticker/24hr"
    try:
        r = requests.get(API_BASE + endpoint)
        r.raise_for_status()
        data = r.json().get("data", {})

        df_ticker = pd.json_normalize(data)

        return df_ticker

    except Exception as e:
        print("❌ Erro ao consultar ticker:", e)
        return None


def pegar_candlesticks(symbol, interval, limit):
    endpoint = f"/rest/v3/markets/{symbol}/candlesticks"
    params = {"interval": interval, "limit": limit}

    response = requests.get(API_BASE + endpoint, params=params)
    if response.status_code != 200:
        print(f"❌ Erro ao obter candlesticks. Status code: {response.status_code}")
        print("Resposta bruta:", response.text)
        raise Exception("Erro na API ao obter candlesticks.")

    candles = response.json() # lista de listas

    df = pd.DataFrame(candles, columns=[
        "timestamp_open", "open", "high", "low", "close",
        "timestamp_close", "volume", "quoteVolume", "count",
        "takerBuyVolume", "takerBuyQuoteVolume"
    ])

    # Converter timestamps para datetime legível
    df["timestamp_open"] = pd.to_datetime(df["timestamp_open"], unit='ms')
    df["timestamp_close"] = pd.to_datetime(df["timestamp_close"], unit='ms')

    return df


def obter_orderbook(symbol):
    """Obtém o livro de ordens (orderbook) para o par informado."""
    endpoint = f"/rest/v3/markets/{symbol}/orderbook"

    params = {"level": 5}

    try:
        r = requests.get(API_BASE + endpoint)
        r.raise_for_status()
        data = r.json()

        # Transformando asks em DataFrame
        df_asks = pd.DataFrame(data['asks'], columns=['price', 'volume'])
        df_asks['type'] = 'ask'

        # Transformando bids em DataFrame
        df_bids = pd.DataFrame(data['bids'], columns=['price', 'volume'])
        df_bids['type'] = 'bid'

        df_order_book = pd.concat([df_asks, df_bids], ignore_index=True)

        return df_order_book

    except Exception as e:
        print("❌ Erro ao consultar orderbook:", e)
        return None


In [ ]:
%%writefile market_analysis.py

import pandas as pd
import numpy as np

def resumir_candles(df, n=20):
    df = df.tail(n).copy()
    df["close"] = pd.to_numeric(df["close"])

    preco_atual = df.iloc[-1]["close"]
    preco_inicial = df.iloc[0]["close"]

    retorno_pct = ((preco_atual - preco_inicial) / preco_inicial) * 100

    volatilidade_pct = df["close"].pct_change().std() * 100

    # RSI 14
    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Handle cases where loss.rolling(14).mean() might be zero or NaN
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = np.where(avg_loss == 0, np.inf, avg_gain / avg_loss)
    rsi = 100 - (100 / (1 + rs))
    rsi_atual = rsi.iloc[-1]

    # SMA 20
    sma_20 = df["close"].rolling(20).mean().iloc[-1] if len(df) >= 20 else np.nan
    distancia_sma_pct = None
    if not pd.isna(sma_20):
        distancia_sma_pct = ((preco_atual - sma_20) / sma_20) * 100

    # Tendência simples (descritiva)
    if retorno_pct > 0.2:
        tendencia = "alta"
    elif retorno_pct < -0.2:
        tendencia = "baixa"
    else:
        tendencia = "lateral"

    return {
        "preco_atual": float(preco_atual),
        "retorno_curto_pct": round(retorno_pct, 3),
        "volatilidade_pct": round(volatilidade_pct, 3),
        "tendencia_curta": tendencia,
        "rsi_14": None if pd.isna(rsi_atual) else round(float(rsi_atual), 2),
        "distancia_sma_20_pct": None if distancia_sma_pct is None else round(float(distancia_sma_pct), 3)
    }

def resumir_ticker_df(df_ticker):
    last = df_ticker.iloc[-1]

    last_price = float(last["last_trade.price"])
    bid_price = float(last["best.bid.price"])
    ask_price = float(last["best.ask.price"])

    spread_pct = ((ask_price - bid_price) / last_price) * 100

    return {
        "preco_atual": last_price,
        "spread_pct": round(spread_pct, 4),
        "variacao_24h_pct": float(last["rolling_24h.price_change_percent"]),
        "volume_24h": float(last["rolling_24h.volume"]),
        "trades_24h": int(last["rolling_24h.trades_count"]),
        "max_24h": float(last["rolling_24h.high"]),
        "min_24h": float(last["rolling_24h.low"])
    }

def pressao_bid_ask_df(df_ticker):
    last = df_ticker.iloc[-1]

    bid_vol = float(last["best.bid.volume"])
    ask_vol = float(last["best.ask.volume"])

    if ask_vol == 0:
        razao = None
    else:
        razao = bid_vol / ask_vol

    return {
        "bid_ask_ratio_top": None if razao is None else round(razao, 3)
    }

def resumir_orderbook_df(df_orderbook, top_n=5):
    asks = df_orderbook[df_orderbook["type"] == "ask"].head(top_n)
    bids = df_orderbook[df_orderbook["type"] == "bid"].head(top_n)

    vol_asks = asks["volume"].astype(float).sum()
    vol_bids = bids["volume"].astype(float).sum()

    if vol_asks == 0:
        pressao = None
    else:
        pressao = vol_bids / vol_asks

    return {
        "volume_asks_top": round(float(vol_asks), 6),
        "volume_bids_top": round(float(vol_bids), 6),
        "pressao_compra": None if pressao is None else round(float(pressao), 3)
    }

def resumir_orderbook_ponderado(df_orderbook, preco_atual, top_n=10):
    df = df_orderbook.copy()

    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["volume"] = pd.to_numeric(df["volume"], errors="coerce")

    asks = df[df["type"] == "ask"].head(top_n)
    bids = df[df["type"] == "bid"].head(top_n)

    # Evitar divisão por zero ou valores não numéricos em 'price'
    asks["peso"] = preco_atual / asks["price"].replace(0, np.nan) # replace 0 with NaN to avoid division by zero
    bids["peso"] = bids["price"] / preco_atual

    volume_asks_ponderado = (asks["volume"] * asks["peso"]).sum()
    volume_bids_ponderado = (bids["volume"] * bids["peso"]).sum()

    total = volume_asks_ponderado + volume_bids_ponderado

    return {
        "pressao_compra_ponderada": (
            volume_bids_ponderado / total if total > 0 else None
        )
    }

def montar_contexto_mercado(df_candles, df_ticker, df_orderbook):
    contexto = {}

    contexto.update(resumir_candles(df_candles))
    contexto.update(resumir_ticker_df(df_ticker))
    contexto.update(pressao_bid_ask_df(df_ticker))
    contexto.update(resumir_orderbook_df(df_orderbook))
    # Check if 'preco_atual' exists before calling resumir_orderbook_ponderado
    if "preco_atual" in contexto:
        contexto.update(resumir_orderbook_ponderado(df_orderbook, contexto["preco_atual"])) # Pass preco_atual
    else:
        print("""⚠️ 'preco_atual' not found in context. Cannot calculate 'pressao_compra_ponderada'.""")

    contexto["ativo"] = "BTC/BRL"
    contexto["timeframe"] = "1m"

    return contexto


In [ ]:
%%writefile ai_integration.py

import json
import os
from openai import OpenAI
from dotenv import load_dotenv

# ==============================================================
# 🧠 INTELIGÊNCIA ARTIFICIAL
# ==============================================================

def montar_prompt_ia(contexto):
    prompt = f"""
Você é um analista quantitativo especializado em mercado financeiro,
microestrutura e fluxo de ordens.

Avalie o estado atual do mercado e decida a melhor ação entre:
COMPRAR, VENDER ou ESPERAR.

Contexto atual do mercado:

Ativo: {contexto['ativo']}
Timeframe: {contexto['timeframe']}

Preço atual: {contexto['preco_atual']}
Retorno curto (%): {contexto['retorno_curto_pct']}
Volatilidade (%): {contexto['volatilidade_pct']}
Tendência curta: {contexto['tendencia_curta']}

Variação 24h (%): {contexto['variacao_24h_pct']}
Máxima 24h: {contexto['max_24h']}
Mínima 24h: {contexto['min_24h']}
Volume 24h: {contexto['volume_24h']}
Trades 24h: {contexto['trades_24h']}

Spread (%): {contexto['spread_pct']}

Order Book:
- Volume bids (top): {contexto['volume_bids_top']}
- Volume asks (top): {contexto['volume_asks_top']}
- Bid/Ask ratio: {contexto['bid_ask_ratio_top']}
- Pressão de compra: {contexto['pressao_compra']}
- Pressão de compra ponderada: {contexto['pressao_compra_ponderada']}

Responda exclusivamente no formato JSON abaixo:

{{
  "acao": "COMPRAR | VENDER | ESPERAR",
  "confianca": 0.0,
  "justificativa_curta": ""
}}
"""
    return prompt

def analisar_mercado(contexto_mercado, client):
    prompt_final = montar_prompt_ia(contexto_mercado)

    try:
        response = client.chat.completions.create(
            model="deepseek/deepseek-r1-0528:free",
            messages=[
                {"role": "user", "content": prompt_final}
            ],
            max_tokens=300,
            temperature=0.2
        )

        if response and response.choices and response.choices[0].message:
            return response.choices[0].message.content
        else:
            # Log the full response object for debugging if choices are missing
            print(f"❌ API response did not contain expected choices or message: {response}")
            return json.dumps({"acao": "ESPERAR", "confianca": 0.0, "justificativa_curta": "API response missing choices or message."})
    except Exception as e:
        print(f"❌ Erro ao chamar a API da IA: {e}")
        return json.dumps({"acao": "ESPERAR", "confianca": 0.0, "justificativa_curta": f"Erro na chamada da API da IA: {e}"})


In [ ]:
# ==============================================================
# 🧩 CÉLULA 2 — CONEXÃO PRIVADA COM A API FOXBIT (EXCHANGE)
# ==============================================================

import time
import hmac
import hashlib
import json
import requests
from urllib.parse import urlencode # Necessário para incluir params na assinatura
import pandas as pd
from datetime import datetime

# Certifique-se de que FOXBIT_API_KEY e FOXBIT_API_SECRET estão definidos
API_BASE = "https://api.foxbit.com.br" # Mantenha o base do GitHub

from google.colab import userdata
FOXBIT_KEY = userdata.get('FOXBIT_KEY')
FOXBIT_SECRET = userdata.get('FOXBIT_SECRET')


# --------------------------------------------------------------
# 🔐 Função para gerar assinatura HMAC (Padrao FOXBIT V3)
# --------------------------------------------------------------
def gerar_assinatura(api_secret, method, path, params=None, body=""):
    """Gera a assinatura HMAC e cabeçalhos exigidos pela Foxbit."""

    timestamp = str(int(time.time() * 1000))

    queryString = urlencode(params) if params else ''

    # O rawBody é a string JSON vazia ou o corpo serializado
    rawBody = json.dumps(body) if body else ''

    # Sequência de assinatura: timestamp + METHOD + PATH + QUERYSTRING + BODY
    preHash = f"{timestamp}{method.upper()}{path}{queryString}{rawBody}"

    assinatura = hmac.new(
        api_secret.encode(), preHash.encode(), hashlib.sha256
    ).hexdigest()

    headers = {
        "X-FB-ACCESS-KEY": API_KEY, # Novo nome do header
        "X-FB-ACCESS-TIMESTAMP": timestamp, # Novo nome do header (usando timestamp como nonce)
        "X-FB-ACCESS-SIGNATURE": assinatura, # Novo nome do header
        "Content-Type": "application/json",
    }
    return headers

# --------------------------------------------------------------
# ⚙️ Função genérica para chamadas autenticadas
# --------------------------------------------------------------
# ATENÇÃO: A função agora aceita `params` para a queryString/assinatura
def chamada_api_privada(method, endpoint, payload=None, params=None):
    url = f"{API_BASE}{endpoint}"

    # Passamos o corpo e os parâmetros para a função de assinatura
    headers = gerar_assinatura(API_SECRET, method, endpoint, params, payload)

    try:
        # Usamos `json=payload` para enviar o corpo serializado
        # Usamos `params=params` para incluir a query string na URL
        response = requests.request(method, url, headers=headers, json=payload, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Erro ao acessar API: {e}")
        if hasattr(e, 'response') and e.response is not None:
            # Imprime a resposta para erros 4xx (como 401, 403, 404)
            print("📩 Resposta da API:", e.response.text)
        return None


# --------------------------------------------------------------
# 🧪 TESTE DE CONEXÃO — consulta informações da conta (seguro)
# --------------------------------------------------------------
print("🔄 Testando conexão com API privada da Foxbit...")

# Endpoint correto do GitHub para obter info do usuário/conta
dados_conta = chamada_api_privada("GET", "/rest/v3/me")

if dados_conta:
    print("✅ Autenticação bem-sucedida!")
    print(json.dumps(dados_conta, indent=2))
else:
    print("⚠️ Não foi possível autenticar. Verifique suas chaves, endpoint ou o novo esquema de assinatura.")

🔄 Testando conexão com API privada da Foxbit...
✅ Autenticação bem-sucedida!
{
  "sn": "FUCNAPPZLEWYAB",
  "email": "taijara@gmail.com",
  "level": 20,
  "disabled": false,
  "created_at": "2018-07-10T20:28:00.000Z"
}


In [ ]:
# --------------------------------------------------------------
# 🔍 Consulta de ticker
# --------------------------------------------------------------
def obter_ticker(symbol):
    """Obtém o preço atual de mercado (último preço) para o par informado."""
    endpoint = f"/rest/v3/markets/{symbol}/ticker/24hr"
    try:
        r = requests.get(API_BASE + endpoint)
        r.raise_for_status()
        #data = r.json()
        data = r.json().get("data", {})
        #print(data)

        last_trade = data[0]["last_trade"]
        rolling = data[0]["rolling_24h"]
        best_ask = data[0]["best"]["ask"]
        best_bid = data[0]["best"]["bid"]

        # Achatar os dicionários internos
        df_ticker = pd.json_normalize(data)

        return df_ticker  #last_trade, rolling, best_ask, best_bid

    except Exception as e:
        print("❌ Erro ao consultar ticker:", e)
        return None


In [ ]:
# --------------------------------------------------------------
# 🔍 Consulta de candles
# --------------------------------------------------------------
def pegar_candlesticks(symbol, interval, limit):
    endpoint = f"/rest/v3/markets/{symbol}/candlesticks"
    params = {"interval": interval, "limit": limit}

    response = requests.get(API_BASE + endpoint, params=params)
    if response.status_code != 200:
        print(f"❌ Erro ao obter candlesticks. Status code: {response.status_code}")
        print("Resposta bruta:", response.text)
        raise Exception("Erro na API ao obter candlesticks.")

    candles = response.json()  # lista de listas

    # Convertendo para DataFrame
    df = pd.DataFrame(candles, columns=[
        "timestamp_open", "open", "high", "low", "close",
        "timestamp_close", "volume", "quoteVolume", "count",
        "takerBuyVolume", "takerBuyQuoteVolume"
    ])

    # Converter timestamps para datetime legível
    df["timestamp_open"] = pd.to_datetime(df["timestamp_open"], unit='ms')
    df["timestamp_close"] = pd.to_datetime(df["timestamp_close"], unit='ms')

    return df


In [ ]:
# --------------------------------------------------------------
# 🔍 Consulta de orderbook
# --------------------------------------------------------------
def obter_orderbook(symbol):
    """Obtém o preço atual de mercado (último preço) para o par informado."""
    endpoint = f"/rest/v3/markets/{symbol}/orderbook"

    params = {"level": 5}  # quantos níveis de cada lado (bid/ask)

    try:
        r = requests.get(API_BASE + endpoint)#, params=params)
        r.raise_for_status()
        data = r.json()
        #print(data)
        #data = r.json().get("data", {})

        # Transformando asks em DataFrame
        df_asks = pd.DataFrame(data['asks'], columns=['price', 'volume'])
        df_asks['type'] = 'ask'  # opcional, para identificar tipo de ordem

        # Transformando bids em DataFrame
        df_bids = pd.DataFrame(data['bids'], columns=['price', 'volume'])
        df_bids['type'] = 'bid'

        # Se quiser, concatenar em um único DataFrame
        df_order_book = pd.concat([df_asks, df_bids], ignore_index=True)

        print(df_order_book)


        return df_order_book

    except Exception as e:
        print("❌ Erro ao consultar ticker:", e)
        return None


In [ ]:
def resumir_candles(df, n=20):
    df = df.tail(n).copy()
    df["close"] = pd.to_numeric(df["close"])

    preco_atual = df.iloc[-1]["close"]
    preco_inicial = df.iloc[0]["close"]

    retorno_pct = ((preco_atual - preco_inicial) / preco_inicial) * 100

    volatilidade_pct = df["close"].pct_change().std() * 100

    # RSI 14
    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    rs = gain.rolling(14).mean() / loss.rolling(14).mean()
    rsi = 100 - (100 / (1 + rs))
    rsi_atual = rsi.iloc[-1]

    # SMA 20
    sma_20 = df["close"].rolling(20).mean().iloc[-1]
    distancia_sma_pct = None
    if not pd.isna(sma_20):
        distancia_sma_pct = ((preco_atual - sma_20) / sma_20) * 100

    # Tendência simples (descritiva)
    if retorno_pct > 0.2:
        tendencia = "alta"
    elif retorno_pct < -0.2:
        tendencia = "baixa"
    else:
        tendencia = "lateral"

    return {
        "preco_atual": float(preco_atual),
        "retorno_curto_pct": round(retorno_pct, 3),
        "volatilidade_pct": round(volatilidade_pct, 3),
        "tendencia_curta": tendencia,
        "rsi_14": None if pd.isna(rsi_atual) else round(float(rsi_atual), 2),
        "distancia_sma_20_pct": None if distancia_sma_pct is None else round(float(distancia_sma_pct), 3)
    }


In [ ]:
def resumir_ticker(ticker):
    last = float(ticker["last"])
    bid = float(ticker["bestBid"])
    ask = float(ticker["bestAsk"])

    spread_pct = ((ask - bid) / last) * 100

    return {
        "preco_atual": last,
        "spread_pct": round(spread_pct, 4),
        "variacao_24h_pct": float(ticker.get("priceChangePercent", 0))
    }


In [ ]:
def resumir_ticker_df(df_ticker):
    last = df_ticker.iloc[-1]

    last_price = float(last["last_trade.price"])
    bid_price = float(last["best.bid.price"])
    ask_price = float(last["best.ask.price"])

    spread_pct = ((ask_price - bid_price) / last_price) * 100

    return {
        "preco_atual": last_price,
        "spread_pct": round(spread_pct, 4),
        "variacao_24h_pct": float(last["rolling_24h.price_change_percent"]),
        "volume_24h": float(last["rolling_24h.volume"]),
        "trades_24h": int(last["rolling_24h.trades_count"]),
        "max_24h": float(last["rolling_24h.high"]),
        "min_24h": float(last["rolling_24h.low"])
    }


In [ ]:
def pressao_bid_ask_df(df_ticker):
    last = df_ticker.iloc[-1]

    bid_vol = float(last["best.bid.volume"])
    ask_vol = float(last["best.ask.volume"])

    if ask_vol == 0:
        razao = None
    else:
        razao = bid_vol / ask_vol

    return {
        "bid_ask_ratio_top": None if razao is None else round(razao, 3)
    }


In [ ]:
def resumir_orderbook(orderbook, profundidade=5):
    bids = orderbook["bids"][:profundidade]
    asks = orderbook["asks"][:profundidade]

    vol_bids = sum(float(b[1]) for b in bids)
    vol_asks = sum(float(a[1]) for a in asks)

    if vol_asks == 0:
        razao = None
    else:
        razao = vol_bids / vol_asks

    return {
        "volume_bids_top": round(vol_bids, 4),
        "volume_asks_top": round(vol_asks, 4),
        "pressao_compra": None if razao is None else round(razao, 3)
    }


In [ ]:
def resumir_orderbook_df(df_orderbook, top_n=5):
    asks = df_orderbook[df_orderbook["type"] == "ask"].head(top_n)
    bids = df_orderbook[df_orderbook["type"] == "bid"].head(top_n)

    vol_asks = asks["volume"].astype(float).sum()
    vol_bids = bids["volume"].astype(float).sum()

    if vol_asks == 0:
        pressao = None
    else:
        pressao = vol_bids / vol_asks

    return {
        "volume_asks_top": round(float(vol_asks), 6),
        "volume_bids_top": round(float(vol_bids), 6),
        "pressao_compra": None if pressao is None else round(float(pressao), 3)
    }


In [ ]:
def resumir_orderbook_ponderado(df_orderbook, preco_atual, top_n=10):
    df = df_orderbook.copy()

    # 🔑 FORÇAR TIPOS NUMÉRICOS
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["volume"] = pd.to_numeric(df["volume"], errors="coerce")

    asks = df[df["type"] == "ask"].head(top_n)
    bids = df[df["type"] == "bid"].head(top_n)

    asks["peso"] = preco_atual / asks["price"]
    bids["peso"] = bids["price"] / preco_atual

    volume_asks_ponderado = (asks["volume"] * asks["peso"]).sum()
    volume_bids_ponderado = (bids["volume"] * bids["peso"]).sum()

    total = volume_asks_ponderado + volume_bids_ponderado

    return {
        "pressao_compra_ponderada": (
            volume_bids_ponderado / total if total > 0 else None
        )
    }


In [ ]:
def montar_contexto_mercado(df_candles, df_ticker, df_orderbook):
    contexto = {}

    contexto.update(resumir_candles(df_candles))
    contexto.update(resumir_ticker_df(df_ticker))
    contexto.update(pressao_bid_ask_df(df_ticker))
    contexto.update(resumir_orderbook_df(df_orderbook))
    contexto.update(resumir_orderbook_ponderado(df_orderbook, contexto["preco_atual"]))


    contexto["ativo"] = "BTC/BRL"
    contexto["timeframe"] = "1m"

    return contexto


In [ ]:
df_candles = pegar_candlesticks("btcbrl", "1m", "10")
df_ticker = obter_ticker("btcbrl")
df_orderbook = obter_orderbook("btcbrl")

/tmp/ipython-input-3036723401.py:24: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df["timestamp_open"] = pd.to_datetime(df["timestamp_open"], unit='ms')
/tmp/ipython-input-3036723401.py:25: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df["timestamp_close"] = pd.to_datetime(df["timestamp_close"], unit='ms')


        price      volume type
0    412262.0  0.00183683  ask
1    412263.0  0.00954529  ask
2    412264.0   0.0065097  ask
3    412401.0  0.00188494  ask
4    412402.0  0.02245062  ask
..        ...         ...  ...
195  370000.0    0.004469  bid
196  369000.0   0.0001355  bid
197  367500.0   0.0000007  bid
198  365000.0  0.00337021  bid
199  362125.0  0.01521574  bid

[200 rows x 3 columns]


In [ ]:
contexto_mercado = montar_contexto_mercado(
    df_candles,
    df_ticker,
    df_orderbook
)

contexto_mercado


{'preco_atual': 412153.0,
 'retorno_curto_pct': np.float64(-0.189),
 'volatilidade_pct': 0.051,
 'tendencia_curta': 'lateral',
 'rsi_14': None,
 'distancia_sma_20_pct': None,
 'spread_pct': 0.0148,
 'variacao_24h_pct': 0.9577629,
 'volume_24h': 60.42824312,
 'trades_24h': 27415,
 'max_24h': 417725.0,
 'min_24h': 394063.0,
 'bid_ask_ratio_top': 23.561,
 'volume_asks_top': 0.042227,
 'volume_bids_top': 0.225293,
 'pressao_compra': 5.335,
 'pressao_compra_ponderada': np.float64(0.9117388884354135),
 'ativo': 'BTC/BRL',
 'timeframe': '1m'}

In [ ]:
def montar_prompt_ia(contexto):
    prompt = f"""
Você é um analista quantitativo especializado em mercado financeiro,
microestrutura e fluxo de ordens.

Avalie o estado atual do mercado e decida a melhor ação entre:
COMPRAR, VENDER ou ESPERAR.

Contexto atual do mercado:

Ativo: {contexto['ativo']}
Timeframe: {contexto['timeframe']}

Preço atual: {contexto['preco_atual']}
Retorno curto (%): {contexto['retorno_curto_pct']}
Volatilidade (%): {contexto['volatilidade_pct']}
Tendência curta: {contexto['tendencia_curta']}

Variação 24h (%): {contexto['variacao_24h_pct']}
Máxima 24h: {contexto['max_24h']}
Mínima 24h: {contexto['min_24h']}
Volume 24h: {contexto['volume_24h']}
Trades 24h: {contexto['trades_24h']}

Spread (%): {contexto['spread_pct']}

Order Book:
- Volume bids (top): {contexto['volume_bids_top']}
- Volume asks (top): {contexto['volume_asks_top']}
- Bid/Ask ratio: {contexto['bid_ask_ratio_top']}
- Pressão de compra: {contexto['pressao_compra']}
- Pressão de compra ponderada: {contexto['pressao_compra_ponderada']}

Responda exclusivamente no formato JSON abaixo:

{{
  "acao": "COMPRAR | VENDER | ESPERAR",
  "confianca": 0.0,
  "justificativa_curta": ""
}}
"""
    return prompt


In [ ]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

from google.colab import userdata
OPENROUTER_KEY = userdata.get('OPENROUTER_KEY')

# Carrega chave de ambiente
load_dotenv()

# Configura cliente para OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_KEY
)




In [ ]:
def analisar_mercado(contexto_mercado):
    prompt_final = montar_prompt_ia(contexto_mercado)

    response = client.chat.completions.create(
        model="deepseek/deepseek-r1-0528:free",
        messages=[
            {"role": "user", "content": prompt_final}
        ],
        max_tokens=300,
        temperature=0.2
    )
    print(response)

    resposta_texto = response.choices[0].message.content

    return resposta_texto


In [ ]:
decisao = analisar_mercado(contexto_mercado)
print(decisao)


ChatCompletion(id=None, choices=None, created=None, model=None, object=None, service_tier=None, system_fingerprint=None, usage=None, error={'message': 'Network connection lost.', 'code': 502, 'metadata': {'provider_name': 'ModelRun'}}, user_id='user_2vaIrBMBTJL5z7k0a3EhArhPXYo')


TypeError: 'NoneType' object is not subscriptable